In [4]:
import pandas as pd
data = pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
data.shape

(10000, 14)

In [6]:
data.columns

Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='object')

In [7]:
data.drop(columns=['RowNumber', 'CustomerId', 'Surname'], inplace=True)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [8]:
data.Geography.unique()

array(['France', 'Spain', 'Germany'], dtype=object)

In [9]:
data.isnull().sum()

CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [10]:
data.describe()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


In [11]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import numpy as np

In [12]:
one_hot_encoder_geo = OneHotEncoder(sparse_output=False, drop="first", dtype=int)
encoded = one_hot_encoder_geo.fit_transform(data[["Geography", "Gender"]])
geo_encoded = pd.DataFrame(data=encoded, columns=one_hot_encoder_geo.get_feature_names_out())
geo_encoded

,Geography_Germany,Geography_Spain,Gender_Male
0,0,0,0
1,0,1,0
2,0,0,0
3,0,0,0
4,0,1,0
...,...,...,...
9995,0,0,1
9996,0,0,1
9997,0,0,0
9998,1,0,1


In [13]:
data = pd.concat([data.drop(columns=["Geography", "Gender"], axis=1), geo_encoded], axis=1)
data.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,0,0,0
1,608,41,1,83807.86,1,0,1,112542.58,0,0,1,0
2,502,42,8,159660.80,3,1,0,113931.57,1,0,0,0
3,699,39,1,0.00,2,0,0,93826.63,0,0,0,0
4,850,43,2,125510.82,1,1,1,79084.10,0,0,1,0


In [14]:
import joblib
standard_scaler = StandardScaler()
data[["Balance", "EstimatedSalary", "CreditScore", "Age"]] = standard_scaler.fit_transform(data[["Balance", "EstimatedSalary", "CreditScore", "Age"]])
joblib.dump(standard_scaler, "scaler.pkl")

['scaler.pkl']

In [15]:
x = data.drop(columns=["Exited"])
y = data["Exited"]

In [16]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=42, stratify=y)

In [17]:
x_train.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male
5866,0.874005,1.342407,8,0.759035,2,0,1,1.223572,0,0,0
1938,-1.371246,-0.087897,3,0.231943,1,0,1,1.081845,0,0,1
4194,-0.812520,1.437761,9,0.334913,1,1,1,1.663809,1,0,0
6332,-0.326221,-0.373958,4,0.223213,1,1,1,-1.383035,0,0,0
1,-0.440036,0.198164,1,0.117350,1,0,1,0.216534,0,1,0


In [18]:
x_test.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male
2605,-0.160673,-0.469311,5,0.355763,2,1,0,-1.466885,0,1,0
9717,1.101634,-0.850726,3,1.104438,1,0,1,1.708485,0,1,1
68,0.108343,-0.373958,5,1.189847,2,0,1,0.235910,1,0,0
9397,1.194755,1.247053,7,0.256835,2,0,1,-0.589429,0,0,0
4004,-1.247084,0.198164,4,-1.225848,2,0,1,0.826264,0,1,0


In [19]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()
lr.fit(x_train, y_train)
lr.score(x_test, y_test)

0.8092

In [20]:
from sklearn import svm
svm = svm.SVC()
svm.fit(x_train,y_train)
svm.score(x_test, y_test)

0.8188

In [21]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier()
knn.fit(x_train,y_train)
knn.score(x_test, y_test)

0.8356

In [22]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier()
dt.fit(x_train, y_train)
dt.score(x_test, y_test)

0.796

In [23]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()
rf.fit(x_train,y_train)
rf.score(x_test, y_test)

0.8664

In [24]:
from sklearn.ensemble import GradientBoostingClassifier
gbc = GradientBoostingClassifier()
gbc.fit(x_train, y_train)
gbc.score(x_test, y_test)

0.8744

In [183]:
import joblib
joblib.dump(gbc, "gbc_model.pkl")

['gbc_model.pkl']